# 01 — Data download

Downloads NHANES files for the self-report vs biomarker comparison.

**Two analytic frames:**

1. **Awareness/treatment/control cascade (2013-2018 pooled).** Self-report items, treatment items, and the matched biomarker for each of seven risks: high cholesterol, hypertension, diabetes, BMI, kidney dysfunction, liver stiffness, and smoking. Uses the standard continuous-NHANES 2-year cycles `_H`, `_I`, `_J`.
2. **Time-trend (1999-2018, 10 cycles).** Self-report items only (`MCQ160B/E/F`) for the three cardiovascular conditions without a measured biomarker in NHANES: heart failure, AMI, stroke.

Liver stiffness lives only in the 2017–March 2020 pre-pandemic release (`P_LUX`), so it gets its own analytic parquet from that cycle.

All downloads cache to `../data/raw/nhanes/`; analytic parquets land in `../data/derived/`.

In [1]:
import os, hashlib, requests
import numpy as np
import pandas as pd
import pyreadstat

DATA = os.path.abspath(os.path.join('..', 'data'))
RAW = os.path.join(DATA, 'raw', 'nhanes')
DERIVED = os.path.join(DATA, 'derived')
os.makedirs(DERIVED, exist_ok=True)

NHANES_BASE = 'https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public'

def download(url, dest):
    if os.path.exists(dest):
        return dest
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    r = requests.get(url, timeout=180)
    if r.status_code != 200:
        return None
    with open(dest, 'wb') as f:
        f.write(r.content)
    return dest

def read_xpt(path, cols=None):
    if path is None or not os.path.exists(path):
        return None
    try:
        df, _ = pyreadstat.read_xport(path)
    except UnicodeDecodeError:
        df, _ = pyreadstat.read_xport(path, encoding='latin1')
    if cols:
        df = df[[c for c in cols if c in df.columns]]
    return df

## Cycle definitions and download helpers

In [2]:
CYCLES = [
    {'label': '1999-2000', 'year': 1999, 'suffix': ''},
    {'label': '2001-2002', 'year': 2001, 'suffix': '_B'},
    {'label': '2003-2004', 'year': 2003, 'suffix': '_C'},
    {'label': '2005-2006', 'year': 2005, 'suffix': '_D'},
    {'label': '2007-2008', 'year': 2007, 'suffix': '_E'},
    {'label': '2009-2010', 'year': 2009, 'suffix': '_F'},
    {'label': '2011-2012', 'year': 2011, 'suffix': '_G'},
    {'label': '2013-2014', 'year': 2013, 'suffix': '_H'},
    {'label': '2015-2016', 'year': 2015, 'suffix': '_I'},
    {'label': '2017-2018', 'year': 2017, 'suffix': '_J'},
]

def fname(component, suffix):
    # 2013-2014 cotinine file is renamed COTNAL_H (adds hydroxycotinine)
    if component == 'COT' and suffix == '_H':
        return 'COTNAL_H.xpt'
    return f'{component}{suffix}.xpt'

def furl(component, year, suffix):
    return f'{NHANES_BASE}/{year}/DataFiles/{fname(component, suffix)}'

def cdir(label):
    return os.path.join(RAW, label.replace('-', '_'))

## Time-trend downloads: DEMO + MCQ across 1999-2018

In [3]:
print('Time-trend (DEMO+MCQ, 10 cycles):')
for c in CYCLES:
    for comp in ['DEMO', 'MCQ']:
        dest = os.path.join(cdir(c['label']), fname(comp, c['suffix']))
        result = download(furl(comp, c['year'], c['suffix']), dest)
        if result is None:
            print(f'  ! missing: {fname(comp, c["suffix"])} for {c["label"]}')
print('done.')

Time-trend (DEMO+MCQ, 10 cycles):


done.


## Comparison cascade downloads: full file set for 2013-2018

In [4]:
COMP_CYCLES = [c for c in CYCLES if c['label'] in ('2013-2014', '2015-2016', '2017-2018')]
COMP_FILES = ['BPQ', 'DIQ', 'KIQ_U', 'SMQ', 'WHQ',
              'BMX', 'BPX', 'TCHOL', 'TRIGLY', 'GLU',
              'BIOPRO', 'COT']

print('Comparison cascade (12 files × 3 cycles):')
for c in COMP_CYCLES:
    for comp in COMP_FILES:
        dest = os.path.join(cdir(c['label']), fname(comp, c['suffix']))
        result = download(furl(comp, c['year'], c['suffix']), dest)
        if result is None:
            print(f'  ! missing: {fname(comp, c["suffix"])}')
print('done.')

Comparison cascade (12 files × 3 cycles):


  ! missing: COTNAL_H.xpt


done.


## P_2017-2020 pre-pandemic release (for liver stiffness)

In [5]:
P_DIR = cdir('P_2017_2020')
P_BASE = f'{NHANES_BASE}/2017/DataFiles'
for comp in ['DEMO', 'MCQ', 'LUX', 'BIOPRO']:
    dest = os.path.join(P_DIR, f'P_{comp}.xpt')
    result = download(f'{P_BASE}/P_{comp}.xpt', dest)
    if result is None:
        print(f'  ! missing: P_{comp}.xpt')
print('P_ release done.')

P_ release done.


## Build time-trend parquet (10 cycles × DEMO+MCQ)

In [6]:
SR_VARS = {'MCQ160B': 'HF_SELF', 'MCQ160E': 'AMI_SELF', 'MCQ160F': 'STROKE_SELF'}

trend_frames = []
for c in CYCLES:
    cd = cdir(c['label'])
    demo = read_xpt(os.path.join(cd, fname('DEMO', c['suffix'])))
    mcq = read_xpt(os.path.join(cd, fname('MCQ', c['suffix'])),
                   cols=['SEQN'] + list(SR_VARS))
    if demo is None or mcq is None:
        continue
    d = demo.merge(mcq, on='SEQN', how='left')
    d = d[d['RIDSTATR'] == 2]
    d = d[d['RIDAGEYR'] >= 20].copy()
    d['AGE'] = d['RIDAGEYR']
    d['FEMALE'] = (d['RIAGENDR'] == 2).astype(int)
    d['SEX'] = d['FEMALE'].map({0: 'Male', 1: 'Female'})
    d['CYCLE'] = c['label']
    wt_col = 'WTMEC2YR' if 'WTMEC2YR' in d.columns else 'WTMEC4YR'
    d['weight'] = d[wt_col]
    for mcq_var, name in SR_VARS.items():
        d[name] = d[mcq_var].map({1.0: 1.0, 2.0: 0.0})
    keep = ['SEQN', 'CYCLE', 'AGE', 'SEX', 'FEMALE', 'weight',
            'SDMVPSU', 'SDMVSTRA'] + list(SR_VARS.values())
    trend_frames.append(d[[k for k in keep if k in d.columns]])

trend = pd.concat(trend_frames, ignore_index=True)
trend.to_parquet(os.path.join(DERIVED, 'sr_trend_1999_2018.parquet'))
print(f'time-trend parquet: {len(trend):,} adults across {trend.CYCLE.nunique()} cycles')
print(trend.groupby('CYCLE').size().to_string())

time-trend parquet: 52,398 adults across 10 cycles
CYCLE
1999-2000    4444
2001-2002    5027
2003-2004    4742
2005-2006    4773
2007-2008    5707
2009-2010    6059
2011-2012    5319
2013-2014    5588
2015-2016    5474
2017-2018    5265


## Build comparison parquet (2013-2018 pooled)

Variables:
- **Hypertension**: BPQ020 (self-report), BPQ050A (treatment), SBP from BPX (biomarker)
- **High cholesterol**: BPQ080, BPQ100D, LBXTC + LBDLDL
- **Diabetes**: DIQ010, DIQ050/DIQ070 (insulin/oral), LBXGLU
- **BMI**: WHD010/WHD020 (self-report), no treatment, BMXBMI (measured)
- **Kidney**: KIQ022, KIQ025 (dialysis), eGFR (CKD-EPI 2021)
- **Liver**: MCQ160L (self-report), no treatment, LSM via P_LUX (separate parquet)
- **Smoking**: SMQ020+SMQ040, no treatment, LBXCOT

In [7]:
def ckd_epi_2021(scr, age, female):
    if pd.isna(scr) or pd.isna(age) or pd.isna(female):
        return np.nan
    kappa = 0.7 if female else 0.9
    alpha = -0.241 if female else -0.302
    sex_factor = 1.012 if female else 1.0
    ratio = scr / kappa
    return 142 * (min(ratio, 1) ** alpha) * (max(ratio, 1) ** -1.200) * (0.9938 ** age) * sex_factor

def smoking_cat(row):
    if row.get('SMQ020') == 2:
        return 'never'
    if row.get('SMQ020') == 1:
        if row.get('SMQ040') in (1, 2):
            return 'current'
        if row.get('SMQ040') == 3:
            return 'former'
    return None

comp_frames = []
for c in COMP_CYCLES:
    cd = cdir(c['label'])
    s = c['suffix']
    demo = read_xpt(os.path.join(cd, fname('DEMO', s)))
    bpq = read_xpt(os.path.join(cd, fname('BPQ', s)),
                   cols=['SEQN', 'BPQ020', 'BPQ050A', 'BPQ080', 'BPQ100D'])
    diq = read_xpt(os.path.join(cd, fname('DIQ', s)),
                   cols=['SEQN', 'DIQ010', 'DIQ050', 'DIQ070'])
    kiq = read_xpt(os.path.join(cd, fname('KIQ_U', s)),
                   cols=['SEQN', 'KIQ022', 'KIQ025'])
    mcq = read_xpt(os.path.join(cd, fname('MCQ', s)),
                   cols=['SEQN', 'MCQ160L', 'MCQ160B', 'MCQ160E', 'MCQ160F'])
    smq = read_xpt(os.path.join(cd, fname('SMQ', s)),
                   cols=['SEQN', 'SMQ020', 'SMQ040'])
    whq = read_xpt(os.path.join(cd, fname('WHQ', s)),
                   cols=['SEQN', 'WHD010', 'WHD020'])
    bmx = read_xpt(os.path.join(cd, fname('BMX', s)),
                   cols=['SEQN', 'BMXBMI'])
    bpx = read_xpt(os.path.join(cd, fname('BPX', s)),
                   cols=['SEQN', 'BPXSY1', 'BPXSY2', 'BPXSY3', 'BPXSY4'])
    tchol = read_xpt(os.path.join(cd, fname('TCHOL', s)),
                     cols=['SEQN', 'LBXTC'])
    trig = read_xpt(os.path.join(cd, fname('TRIGLY', s)),
                    cols=['SEQN', 'LBDLDL', 'WTSAF2YR'])
    glu = read_xpt(os.path.join(cd, fname('GLU', s)),
                   cols=['SEQN', 'LBXGLU'])
    bio = read_xpt(os.path.join(cd, fname('BIOPRO', s)),
                   cols=['SEQN', 'LBXSCR'])
    cot = read_xpt(os.path.join(cd, fname('COT', s)),
                   cols=['SEQN', 'LBXCOT'])

    d = demo
    for sub in [bpq, diq, kiq, mcq, smq, whq, bmx, bpx, tchol, trig, glu, bio, cot]:
        if sub is not None:
            d = d.merge(sub, on='SEQN', how='left')

    sy_cols = [c2 for c2 in ['BPXSY1', 'BPXSY2', 'BPXSY3', 'BPXSY4'] if c2 in d.columns]
    if sy_cols:
        d['SBP_MEAN'] = d[sy_cols].replace(0, np.nan).mean(axis=1)
    else:
        d['SBP_MEAN'] = np.nan

    # self-reported BMI from WHQ (inches, lbs); 7777/9999 are refused/don't-know
    ht_in = d['WHD010'].where(d['WHD010'] < 7000)
    wt_lb = d['WHD020'].where(d['WHD020'] < 7000)
    d['BMI_SELF'] = (wt_lb * 0.453592) / ((ht_in * 0.0254) ** 2)

    d = d[d['RIDSTATR'] == 2]
    d = d[d['RIDAGEYR'] >= 20].copy()
    d['AGE'] = d['RIDAGEYR']
    d['FEMALE'] = (d['RIAGENDR'] == 2).astype(int)
    d['SEX'] = d['FEMALE'].map({0: 'Male', 1: 'Female'})
    d['CYCLE'] = c['label']
    d['weight'] = d['WTMEC2YR'] / len(COMP_CYCLES)
    if 'WTSAF2YR' in d.columns:
        d['weight_fasting'] = d['WTSAF2YR'] / len(COMP_CYCLES)
    else:
        d['weight_fasting'] = np.nan

    d['eGFR'] = d.apply(lambda r: ckd_epi_2021(r['LBXSCR'], r['AGE'], r['FEMALE']), axis=1)
    d['SMOKE_CAT'] = d.apply(smoking_cat, axis=1)

    sr_map = {1.0: 1.0, 2.0: 0.0}
    d['HBP_SELF']  = d['BPQ020'].map(sr_map)
    d['HBP_TRT']   = d['BPQ050A'].map(sr_map)
    d['HC_SELF']   = d['BPQ080'].map(sr_map)
    d['HC_TRT']    = d['BPQ100D'].map(sr_map)
    d['DM_SELF']   = d['DIQ010'].map(sr_map)
    d['DM_INS']    = d['DIQ050'].map(sr_map)
    d['DM_PILL']   = d['DIQ070'].map(sr_map)
    d['DM_TRT']    = ((d['DM_INS'] == 1) | (d['DM_PILL'] == 1)).astype(float)
    d.loc[d['DM_SELF'].isna(), 'DM_TRT'] = np.nan
    d['KID_SELF']  = d['KIQ022'].map(sr_map)
    d['DIALYSIS']  = d['KIQ025'].map(sr_map)
    d['LIVER_SELF']= d['MCQ160L'].map(sr_map)
    d['HF_SELF']   = d['MCQ160B'].map(sr_map)
    d['AMI_SELF']  = d['MCQ160E'].map(sr_map)
    d['STROKE_SELF']=d['MCQ160F'].map(sr_map)

    keep = ['SEQN', 'CYCLE', 'AGE', 'SEX', 'FEMALE',
            'weight', 'weight_fasting', 'SDMVPSU', 'SDMVSTRA',
            'BMXBMI', 'BMI_SELF',
            'SBP_MEAN', 'HBP_SELF', 'HBP_TRT',
            'LBXTC', 'LBDLDL', 'HC_SELF', 'HC_TRT',
            'LBXGLU', 'DM_SELF', 'DM_TRT', 'DM_INS', 'DM_PILL',
            'eGFR', 'LBXSCR', 'KID_SELF', 'DIALYSIS',
            'LIVER_SELF', 'HF_SELF', 'AMI_SELF', 'STROKE_SELF',
            'SMOKE_CAT', 'LBXCOT']
    comp_frames.append(d[[k for k in keep if k in d.columns]])

comp = pd.concat(comp_frames, ignore_index=True)
comp.to_parquet(os.path.join(DERIVED, 'sr_comparison_2013_2018.parquet'))
print(f'comparison parquet: {len(comp):,} adults pooled across {len(COMP_CYCLES)} cycles')

comparison parquet: 16,327 adults pooled across 3 cycles


## Build LSM parquet from P_2017-2020

In [8]:
p_demo = read_xpt(os.path.join(P_DIR, 'P_DEMO.xpt'))
p_mcq = read_xpt(os.path.join(P_DIR, 'P_MCQ.xpt'),
                 cols=['SEQN', 'MCQ160L'])
p_lux = read_xpt(os.path.join(P_DIR, 'P_LUX.xpt'),
                 cols=['SEQN', 'LUXSMED', 'LUXSIQR'])

p = p_demo.merge(p_mcq, on='SEQN', how='left').merge(p_lux, on='SEQN', how='left')
p = p[p['RIDSTATR'] == 2]
p = p[p['RIDAGEYR'] >= 20].copy()
p['AGE'] = p['RIDAGEYR']
p['FEMALE'] = (p['RIAGENDR'] == 2).astype(int)
p['SEX'] = p['FEMALE'].map({0: 'Male', 1: 'Female'})
wt_col = 'WTMECPRP' if 'WTMECPRP' in p.columns else 'WTMEC2YR'
p['weight'] = p[wt_col]
p['LIVER_SELF'] = p['MCQ160L'].map({1.0: 1.0, 2.0: 0.0})
p['LSM_VALID'] = ((p['LUXSIQR'] / p['LUXSMED']) <= 0.30)

keep = ['SEQN', 'AGE', 'SEX', 'FEMALE', 'weight',
        'SDMVPSU', 'SDMVSTRA',
        'LUXSMED', 'LSM_VALID', 'LIVER_SELF']
p[keep].to_parquet(os.path.join(DERIVED, 'sr_lsm_p2017_2020.parquet'))
print(f'LSM parquet: {len(p):,} adults from P_2017-2020 release')
print(f'  with LSM measured: {p["LUXSMED"].notna().sum():,}')
print(f'  with valid LSM (IQR/median ≤ 0.30): {p["LSM_VALID"].sum():,}')

LSM parquet: 8,544 adults from P_2017-2020 release
  with LSM measured: 7,923
  with valid LSM (IQR/median ≤ 0.30): 7,707
